## 1. Imports

In [ ]:
from __future__ import annotations
import random
import copy
import time
import numpy as np
from pathlib import Path

from deap import base, creator, tools

# DRY imports
from src.notebooks.data_loader import load_data
from src.notebooks.population import create_random_individual
from src.notebooks.operators import course_aware_crossover, smart_mutation
from src.notebooks.evaluation import create_evaluator, get_constraint_breakdown
from src.notebooks.evolution import EvolutionConfig, setup_deap, get_best_individual, EvolutionStats
from src.notebooks.visualization import plot_convergence, plot_constraint_breakdown, print_summary
from src.notebooks.rl_helper import SimpleRLSelector, load_trained_agent

print(" All imports successful!")

## 2. Mode E Configuration

In [ ]:
from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters
POP_SIZE = 50
NGEN = 100
CXPB = 0.9
MUTPB = 0.2
FITNESS_WEIGHTS = (-1.0, -0.01)

# MODE E SPECIFIC: RL parameters
REPAIR_PROB = 0.3
RL_LEARNING_RATE = 0.1
RL_DISCOUNT = 0.95
RL_EPSILON = 0.3         # Initial exploration rate
RL_EPSILON_DECAY = 0.99  # Decay per generation
RL_MIN_EPSILON = 0.05

# Paths - Organized by mode with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_e_rl_guided/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Mode E Config: pop={POP_SIZE}, ngen={NGEN}, epsilon={RL_EPSILON}")
print(f"📁 Output: {OUTPUT_DIR}")

## 3. Load Data & Check for Trained Models

In [ ]:
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)
evaluate = create_evaluator(data)

print(f" {data.summary()}")

# Check for pre-trained model
trained_model = load_trained_agent("../models/rl_agents")
if trained_model:
    print(f" Found trained model (can use for inference)")
else:
    print(f" No pre-trained model found, will use online Q-learning")

## 4. Test RL Selector

In [ ]:
# Test RL selector
rl_selector = SimpleRLSelector(
    learning_rate=RL_LEARNING_RATE,
    discount=RL_DISCOUNT,
    epsilon=RL_EPSILON,
    epsilon_decay=RL_EPSILON_DECAY,
    min_epsilon=RL_MIN_EPSILON,
)

test_ind = create_random_individual(data)
print(f"Initial fitness: hard={evaluate(test_ind)[0]}")

for _ in range(5):
    action, fixes, reward = rl_selector.apply(test_ind, data, evaluate)
    print(f"  Action: {action}, fixes={fixes}, reward={reward:.1f}")

print(f"After RL: hard={evaluate(test_ind)[0]}")
print(f"Q-table states: {len(rl_selector.q_table)}")

## 5. RL-Guided NSGA-II Evolution (Mode E)

In [ ]:
def run_rl_guided_nsga2():
    """Run NSGA-II with RL-guided heuristic selection."""
    print(f" RL-Guided NSGA-II: pop={POP_SIZE}, ngen={NGEN}")
    start = time.time()
    
    setup_deap(FITNESS_WEIGHTS)
    
    # Initialize RL selector (fresh Q-table)
    rl_selector = SimpleRLSelector(
        learning_rate=RL_LEARNING_RATE,
        discount=RL_DISCOUNT,
        epsilon=RL_EPSILON,
        epsilon_decay=RL_EPSILON_DECAY,
        min_epsilon=RL_MIN_EPSILON,
    )
    
    toolbox = base.Toolbox()
    toolbox.register("individual", lambda: creator.Individual(create_random_individual(data)))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", course_aware_crossover)
    toolbox.register("mutate", lambda ind: smart_mutation(ind, data))
    toolbox.register("select", tools.selNSGA2)
    
    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    
    stats = EvolutionStats()
    total_rewards = 0.0
    epsilon_history = []
    reward_history = []
    
    for gen in range(NGEN):
        offspring = [copy.deepcopy(ind) for ind in toolbox.select(pop, len(pop))]
        
        # Crossover
        for i in range(0, len(offspring)-1, 2):
            if random.random() < CXPB:
                toolbox.mate(offspring[i], offspring[i+1])
                del offspring[i].fitness.values
                del offspring[i+1].fitness.values
        
        # Mutation
        for ind in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(ind)
                del ind.fitness.values
        
        # === MODE E SPECIFIC: RL-Guided Repair ===
        gen_reward = 0.0
        for ind in offspring:
            if random.random() < REPAIR_PROB:
                genes = list(ind)
                _, _, reward = rl_selector.apply(genes, data, evaluate)
                gen_reward += reward
                ind[:] = genes
                del ind.fitness.values
        
        total_rewards += gen_reward
        reward_history.append(gen_reward)
        epsilon_history.append(rl_selector.epsilon)
        
        # Decay epsilon
        rl_selector.decay_epsilon()
        
        # Evaluate
        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)
        
        pop = toolbox.select(pop + offspring, POP_SIZE)
        
        # Stats
        hard_vals = [ind.fitness.values[0] for ind in pop]
        soft_vals = [ind.fitness.values[1] for ind in pop]
        stats.generations.append(gen)
        stats.min_hard.append(float(min(hard_vals)))
        stats.avg_hard.append(float(np.mean(hard_vals)))
        stats.max_hard.append(float(max(hard_vals)))
        stats.feasible_count.append(sum(1 for h in hard_vals if h == 0))
        stats.min_soft.append(float(min(soft_vals)))
        stats.avg_soft.append(float(np.mean(soft_vals)))
        
        if gen % 20 == 0 or gen == NGEN-1:
            print(f"  Gen {gen:3d}: min_hard={stats.min_hard[-1]:3.0f}, ε={rl_selector.epsilon:.3f}, reward={gen_reward:.1f}")
    
    stats.elapsed_time = time.time() - start
    print(f" Done in {stats.elapsed_time:.1f}s")
    print(f"RL Stats: {rl_selector.get_stats()}")
    return pop, stats, rl_selector, epsilon_history, reward_history

final_pop, stats, rl_selector, epsilon_history, reward_history = run_rl_guided_nsga2()

## 6. Results

In [ ]:
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

print_summary(final_pop, stats, breakdown)

plot_convergence(stats, OUTPUT_DIR / "mode_e_convergence.png", title_prefix="Mode E: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_e_breakdown.png", title="Mode E: Constraint Violations")

## 7. RL Learning Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Epsilon decay
ax1 = axes[0]
ax1.plot(epsilon_history, 'b-', linewidth=2)
ax1.set_xlabel("Generation")
ax1.set_ylabel("Epsilon (Exploration Rate)")
ax1.set_title("Mode E: Epsilon Decay")
ax1.grid(True, alpha=0.3)

# Rewards
ax2 = axes[1]
ax2.plot(reward_history, 'g-', alpha=0.5, linewidth=1)
# Rolling average
window = 10
if len(reward_history) >= window:
    rolling = np.convolve(reward_history, np.ones(window)/window, mode='valid')
    ax2.plot(range(window-1, len(reward_history)), rolling, 'g-', linewidth=2, label=f'{window}-gen avg')
ax2.set_xlabel("Generation")
ax2.set_ylabel("Episode Reward")
ax2.set_title("Mode E: RL Rewards")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mode_e_rl_learning.png", dpi=150)
plt.show()

## 8. Q-Table Analysis

In [ ]:
# Show learned Q-values for most visited states
print("\n Learned Q-Values (top states):")
print("="*60)

# Sort states by number of visits (approximate via sum of Q-value magnitudes)
state_importance = []
for state, actions in rl_selector.q_table.items():
    importance = sum(abs(v) for v in actions.values())
    state_importance.append((state, actions, importance))

state_importance.sort(key=lambda x: x[2], reverse=True)

for state, actions, _ in state_importance[:5]:
    print(f"\nState: {state}")
    for action, q_val in sorted(actions.items(), key=lambda x: x[1], reverse=True):
        best_marker = "" if q_val == max(actions.values()) else "  "
        print(f"  {best_marker} {action}: {q_val:.2f}")